**Выгрузка численности населения с сайта Росстата Иркутской области**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

In [3]:
pip install requests beautifulsoup4 pandas openpyxl lxml

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import BytesIO
import re
import math

In [5]:
def rosstat_to_df(file_name, page_name):
    # URL страницы
    url = 'https://38.rosstat.gov.ru/folder/167937'

    try:
        # Получаем содержимое страницы
        response = requests.get(url, timeout=30, verify=False)  # без сертификата
        response.raise_for_status()
    
        # Парсим HTML
        soup = BeautifulSoup(response.content, 'html.parser')
    
        # Ищем ссылку на файл с нужным названием
        target_file = None
        for link in soup.find_all('a', href=True):

            link_text = link.get_text(strip=True)
            href = link['href']
        
            # Проверяем, содержит ли текст ссылки нужное название
            #file_name = 'Численность мужчин и женщин'
            #file_name = 'Иркутская область_2012_2021_с учетом итогов Всероссийской переписи населения 2020 года'
        
            if file_name in href:
                # Формируем полный URL
                if href.startswith('http'):
                    target_file = href
                elif href.startswith('/'):
                    # Если ссылка относительная, добавляем домен
                    base_url = 'https://38.rosstat.gov.ru'
                    target_file = base_url + href
                else:
                    # Если ссылка относительно текущей страницы
                    target_file = url.rstrip('/') + '/' + href
                break
    
        if not target_file:
            raise FileNotFoundError("Файл с указанным названием не найден на странице")
    
        print(f"Найден файл: {target_file}")
    
        # Загружаем Excel‑файл
        file_response = requests.get(target_file, timeout=30, verify=False) # без сертификата
        file_response.raise_for_status()
    
        # Создаём объект BytesIO из содержимого файла
        excel_file = BytesIO(file_response.content)
    
        # Читаем Excel в датасет
        # df = pd.read_excel(excel_file)
    
        #df_population = pd.read_excel(excel_file, engine='openpyxl', sheet_name = page_name)
        df_population = pd.read_excel(excel_file, engine='xlrd', sheet_name = page_name)
        
        print("Файл успешно загружен в датасет!")
    
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при загрузке страницы или файла: {e}")
    except FileNotFoundError as e:
        print(e)
    except Exception as e:
        print(f"Ошибка при обработке файла: {e}")
    return df_population

In [6]:
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

file_name = {
    2016: 'pol_voz_2016(1)_626391.xls',
    2017: 'pol_voz_2017_626392.xls',
    2018: 'pol_voz_2018_626393.xls',
    2019: 'pol_voz_2019_626395.xls',
    2020: 'Численность населения по полу и возрасту на начало 2020 года_626396.xls',
    2021: 'Численность населения по полу и возрасту на начало 2021 года_626398.xls',
    2022: 'Численность населения по полу и возрасту на начало 2021 года_626398.xls',
    2023: 'Численность населения по полу и возрасту на начало 2023 года(1).xls',
    2024: 'Численность населения по полу и возрасту на начало 2024 года..xls'
}


In [7]:
df_population = {}
for year in years:
    if year == 2019:
        page_name = 'Иркутская обл.'
    elif year == 2024:
        page_name = 'Область'
    else:
        page_name = 'Иркутская область'
    df_population[year] = rosstat_to_df(file_name[year], page_name)
    df_population[year].to_csv(f'E:/IrkutskProject/Data/{year}/df_population_{year}.csv')
    #display(df_population[year])

C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Найден файл: https://38.rosstat.gov.ru/storage/mediabank/pol_voz_2016(1)_626391.xls


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Файл успешно загружен в датасет!


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Файл успешно загружен в датасет!


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Найден файл: https://38.rosstat.gov.ru/storage/mediabank/Численность населения по полу и возрасту на начало 2024 года..xls


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Файл успешно загружен в датасет!


In [8]:
for year in years:
    for row_number in range(126, 11, -6):
        df_population[year].drop(df_population[year].index[row_number:row_number + 1], inplace=True)
    df_population[year] = df_population[year].iloc[7:]
    df_population[year] = df_population[year].iloc[:,:5]
    
    df_population[year] =  df_population[year].reset_index(drop=True)
    df_population[year].iloc[100,0] = 100
    
#path = 'E:/IrkutskProject/Data/'
#df_population[2016].to_csv(f'{path}Common/avg_population_2016.csv')

In [9]:
# year = 2016
df = {}

columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]

data = []

for year in years:
    for col_number in range(2, df_population[year].shape[1]):
    
        new_row = []    
        new_row.append(year) # Год
        if (col_number == 2): # Пол
            new_row.append('всего')
        if (col_number == 3):
            new_row.append('мужчины')
        if (col_number == 4):
            new_row.append('женщины')

        #Всего
        value = 0
        for row_number in range(0, df_population[year].shape[0]):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
    
        # 0-4
        value = 0
        for row_number in range(0, 5):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 5-6
        value = 0
        for row_number in range(5, 7): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 7-14
        value = 0
        for row_number in range(7, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
      
        # 15-17
        value = 0
        for row_number in range(15, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 18-24
        value = 0
        for row_number in range(18, 25): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 25-34
        value = 0
        for row_number in range(25, 35): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 35-44
        value = 0
        #for row_number in range(35, 44): 
        for row_number in range(35, 45): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 45-54
        value = 0
        #for row_number in range(45, 54): 
        for row_number in range(45, 55): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 55-64
        value = 0
        #for row_number in range(55, 64): 
        for row_number in range(55, 65):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 65+
        value = 0
        for row_number in range(65, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-14
        value = 0
        for row_number in range(0, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 15+
        value = 0
        for row_number in range(15, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-17
        value = 0
        for row_number in range(0, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)        

        data.append(new_row)
    
df = pd.DataFrame(data, columns=columns)
    

In [10]:
# Промежуточный результат. Абсолютные значения
df

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2412800,184508,68365,231220,74809,196504,411756,346933,294948,318766,284991,484093,1928707,558902
1,2016,мужчины,1115508,94508,34959,118511,38540,98210,207138,165788,136331,132933,88590,247978,867530,286518
2,2016,женщины,1297292,90000,33406,112709,36269,98294,204618,181145,158617,185833,196401,236115,1061177,272384
3,2017,всего,2408901,183080,70049,239152,75149,183862,406890,349587,287933,318660,294539,492281,1916620,567430
4,2017,мужчины,1113729,94230,35690,122656,38747,91663,205544,166899,133131,133218,91951,252576,861153,291323
5,2017,женщины,1295172,88850,34359,116496,36402,92199,201346,182688,154802,185442,202588,239705,1055467,276107
6,2018,всего,2404195,177110,73839,245329,78081,175602,395025,353974,284907,316167,304161,496278,1907917,574359
7,2018,мужчины,1111049,91120,37669,125689,40076,88009,199472,169328,131782,132510,95394,254478,856571,294554
8,2018,женщины,1293146,85990,36170,119640,38005,87593,195553,184646,153125,183657,208767,241800,1051346,279805
9,2019,всего,2397763,170405,74605,253078,80547,172072,379206,358786,283865,312297,312902,498088,1899675,578635


In [11]:
def round_math(num):
    if num >= 0:
        return math.floor(num + 0.5)
    else:
        return math.ceil(num - 0.5)

In [12]:
df_avg = {}
columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]
data = []

for row_number in range(0, df.shape[0] - 3):
    new_row = []
    new_row.append(df.iloc[row_number, 0])
    new_row.append(df.iloc[row_number, 1])
    
    for col_number in range(2,df.shape[1]):
        value = round_math((df.iloc[row_number, col_number] + df.iloc[row_number + 3, col_number])/2)
        new_row.append(value)
    data.append(new_row)
df_avg = pd.DataFrame(data, columns=columns)
result = pd.concat([df_avg, df.iloc[-3:]], ignore_index=True)
display(result)

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2410851,183794,69207,235186,74979,190183,409323,348260,291441,318713,289765,488187,1922664,563166
1,2016,мужчины,1114619,94369,35325,120584,38644,94937,206341,166344,134731,133076,90271,250277,864342,288921
2,2016,женщины,1296232,89425,33883,114603,36336,95247,202982,181917,156710,185638,199495,237910,1058322,274246
3,2017,всего,2406548,180095,71944,242241,76615,179732,400958,351781,286420,317414,299350,494280,1912269,570895
4,2017,мужчины,1112389,92675,36680,124173,39412,89836,202508,168114,132457,132864,93673,253527,858862,292939
5,2017,женщины,1294159,87420,35265,118068,37204,89896,198450,183667,153964,184550,205678,240753,1053407,277956
6,2018,всего,2400979,173758,74222,249204,79314,173837,387116,356380,284386,314232,308532,497183,1903796,576497
7,2018,мужчины,1109440,89448,37978,127607,40624,87221,195507,170847,131440,131825,96947,255032,854409,295655
8,2018,женщины,1291539,84310,36245,121597,38691,86617,191609,185534,152946,182408,211585,242152,1049388,280842
9,2019,всего,2394478,166162,73833,257439,81632,171343,372520,360286,285077,308254,317934,497434,1897045,579066


In [13]:
path = 'E:/IrkutskProject/Data/'
result.to_csv(f'{path}Common/avg_population_2016-2024.csv', index=False)